# Nettoyage corrigé & Résolution de la Question 4
Ce notebook contient les corrections sur les formats de dates et la standardisation des noms de villes, ainsi que l'analyse complète pour la question 4 (Évolution sur 18 mois et allocation budgétaire).

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt

## 1. La fonction de nettoyage vitale (Comptes Test)

In [ ]:
def detect_test(ch):
    if pd.isna(ch):
        return "Missing"
    elif re.search(r"test", str(ch), re.IGNORECASE):
        return "Test"
    else:
        return "Real"

## 2. Chargement et Nettoyage Corrigé

In [ ]:
booking_requests_df = pd.read_csv("booking_requests.csv")
cleaners_df = pd.read_csv("cleaners.csv")
customers_df = pd.read_csv("customers.csv")
marketing_spend_df = pd.read_csv("marketing_spend.csv")

# --- CUSTOMERS ---
customers_df = customers_df.drop_duplicates()
customers_df = customers_df[customers_df["customer_id"].apply(detect_test) == 'Real']
customers_df['signup_date'] = pd.to_datetime(customers_df['signup_date'], errors='coerce')

# --- CLEANERS ---
cleaners_df = cleaners_df.drop_duplicates()
cleaners_df = cleaners_df[cleaners_df["cleaner_id"].apply(detect_test) == 'Real']
for col in ['application_date', 'onboarded_date', 'deactivated_date']:
    cleaners_df[col] = pd.to_datetime(cleaners_df[col], errors='coerce')

# --- BOOKING REQUESTS ---
booking_requests_df = booking_requests_df.drop_duplicates()
for col in ['request_id', 'customer_id', 'cleaner_id']:
    booking_requests_df = booking_requests_df[booking_requests_df[col].apply(detect_test) == 'Real']

# CORRECTION MAJEURE : Les dates sont parfois au format européen (27/02/2026), il faut dayfirst=True
booking_requests_df['request_date'] = pd.to_datetime(booking_requests_df['request_date'], dayfirst=True, errors='coerce')
booking_requests_df = booking_requests_df[booking_requests_df["price_eur"] > 0]

# CORRECTION MAJEURE : Les villes sont mal écrites (lyon, LYON, Lyon 3e, PARIS 11, etc.)
booking_requests_df['city'] = booking_requests_df['city'].astype(str).str.title().str.strip()
booking_requests_df['city'] = booking_requests_df['city'].apply(lambda x: 'Lyon' if 'Lyon' in x else ('Paris' if 'Paris' in x else x))

# --- MARKETING SPEND ---
marketing_spend_df = marketing_spend_df.drop_duplicates()
marketing_spend_df = marketing_spend_df[marketing_spend_df["spend_eur"] > 0]
marketing_spend_df['month'] = pd.to_datetime(marketing_spend_df['month'], dayfirst=True, errors='coerce')
marketing_spend_df['city'] = marketing_spend_df['city'].str.title().str.strip()

## 3. Analyse pour la Question 4 du Board

In [ ]:
booking_requests_df['month_period'] = booking_requests_df['request_date'].dt.to_period('M')
marketing_spend_df['month_period'] = marketing_spend_df['month'].dt.to_period('M')

# Évolution du taux d'échec (Unfulfilled)
status_counts = booking_requests_df.groupby(['month_period', 'city', 'status']).size().unstack(fill_value=0)
status_counts['total_requests'] = status_counts.sum(axis=1)
status_counts['unfulfilled_rate_pct'] = (status_counts['unfulfilled'] / status_counts['total_requests']) * 100

print("--- Taux de requêtes non satisfaites (Unfulfilled %) --- ")
display(status_counts[['unfulfilled_rate_pct']].unstack('city').round(1).tail(6))

# Évolution du budget Marketing
spend_trends = marketing_spend_df.groupby(['month_period', 'side'])['spend_eur'].sum().unstack(fill_value=0)
print("\n--- Budget Marketing (Clients vs Cleaners) ---")
display(spend_trends.tail(6))

# Graphique
lyon_data = status_counts.xs('Lyon', level='city')
plt.figure(figsize=(10, 4))
plt.plot(lyon_data.index.astype(str), lyon_data['unfulfilled_rate_pct'], marker='o', color='red', label="Lyon")
plt.title("Explosion du taux de requêtes non satisfaites à Lyon (Manque de Cleaners)")
plt.ylabel("% Unfulfilled")
plt.xticks(rotation=45)
plt.grid(True)
plt.legend()
plt.show()